## 工具的应用案例

### 案例1：使用args_schema

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.messages import HumanMessage

# 优先加载配置文件
load_dotenv(override=True)

#初始化好要调用的模型，这里选择的是Qwen
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#采用pydantic模型定义，需要定义一个类，用来描述字段
class WeatherSchema(BaseModel):
    city: str = Field(default="北京", description='城市的名称')
    if_forecast: bool = Field(default=False, description="是否包含明天的天气预报")

#如何加文档注释，很关键了.name_or_callable是重新定义了名称, description 方法的注释， args_schema 定义的字段类型
@tool(name_or_callable="get_weather_and_forecast", description="查询当日天气，可以包含明日天气预报", args_schema=WeatherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city} 今天天气不错，晴空万里！"
    if if_forecast:
        res += "\n明天的天气，会有雷阵雨"
    return res

#1. 当前的大模型绑定一个工具get_weather
model_with_tools = model_openai.bind_tools([get_weather])

#2. 维护和生成一个消息列表
messages = [HumanMessage(content="今天深圳天气如何？明天呢？")]

#3. 调用大模型的（带有绑定工具类的），返回：AIMessage
response = model_with_tools.invoke(messages)
#消息里追加 AIMessage
messages.append(response)

#4. 获取响应中的tool_calls字段信息
to_calls = response.tool_calls
#找到名称为get_weather_and_forecast的工具方法，消息里追加ToolMessage
for call in to_calls:
    if  call["name"] == "get_weather_and_forecast":
        # 5. 调用工具（因为大模型不能直接调用工具，所以此时我们主动让工具调用执行）
        # 调用完后，返回：ToolMessage的实例
        tool_message = get_weather.invoke(call)
        messages.append(tool_message)

# model_with_tools = "我这个模型可能会调工具"， model_deepseek = "我这个模型就是纯聊天"
#6. 调用模型: 为了避免触发重复调用工具类，请用model_openai。
final_response = model_openai.invoke(messages)

#7. 添加到消息列表中
messages.append(final_response)

#8. 遍历消息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天深圳天气如何？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_37e85d69df264d158a51baba)
 Call ID: call_37e85d69df264d158a51baba
  Args:
    city: 深圳
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

深圳 今天天气不错，晴空万里！
明天的天气，会有雷阵雨
================================== Ai Message ==================================

根据查询结果，深圳今天天气晴朗，晴空万里。

不过明天可能会有**雷阵雨**，建议您出门前留意最新的天气预报，并准备好雨具。


### 案例2：撰写docstring

In [27]:
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

@tool(name_or_callable="get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str, if_forecast: bool):
    """
    查询当天的天气，可以包含明天的天气预报

    Args:
        city: 城市名称
        if_forecast: 是否包含明天的天气预报

    Returns:
        返回天气预报的内容
    """
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n 明天要下雨"
    return res

#1. 维护一个消息列表 (1、2步骤部分先后)
messages = [HumanMessage(content="今天深圳的天气怎么样？明天呢？")]

#2. 绑定一个工具
model_with_tools = model_openai.bind_tools([get_weather])
rprint(convert_to_openai_tool(get_weather))
#3. 调用工具消息，返回AIMessage
response = model_with_tools.invoke(messages)
messages.append(response)

# 4. 调用工具类
tool_calls = response.tool_calls
for call in tool_calls:
    if call["name"] == "get_weather_and_forecast":
        tool_msg = get_weather.invoke(call)
        #添加工具的消息
        messages.append(tool_msg)

# 5. 这个是对话模型的调用，再次调用后就会采用tool类的。
final_response = model_openai.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()


{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当天的天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称', 'type': 'string'},
                'if_forecast': {'description': '是否包含明天的天气预报', 'type': 'boolean'}
            },
            'required': ['city', 'if_forecast'],
            'type': 'object'
        }
    }
}

================================ Human Message =================================

今天深圳的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_dad3d42ea35b4999b00aa805)
 Call ID: call_dad3d42ea35b4999b00aa805
  Args:
    city: 深圳
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

深圳 今天天气不错
 明天要下雨
================================== Ai Message ==================================

今天深圳的天气不错，适合出行。

不过明天预计会有雨，建议您关注最新的天气变化，出门记得随身带把伞以防万一。


### 案例3：多工具调用

In [24]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.messages import HumanMessage
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

# 1. 定义工具
# 定义股票查询工具
@tool(name_or_callable="get_stock_price_tool", parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """
    获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司，微软公司，谷歌公司）
        timeframe: 时间范围（today-今日，week-本周，month-本月）

    Returns:
        指定公司的股票价格信息，string类型
    """
    #模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 414.58, "week": 413.50, "month": 405.77},
        "谷歌公司": {"today": 135.23, "week": 136.56, "month": 136.78},
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"

# 定义新闻搜索工具
@tool(name_or_callable="search_news_tool", parse_docstring=True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
      "苹果公司": [
          "苹果发布新款iPhone，股价上涨3%",
          "苹果与欧盟达成反垄断和解协议",
          "苹果将在印度扩大生产规模"
      ],
      "微软公司": [
          "微软Azure云业务季度增长超预期",
          "微软完成对Nuance的收购",
          "微软推出新一代AI助手Copilot "
      ],
      "谷歌公司": [
          "谷歌发布新AI模型，性能提升20%",
          "谷歌与OpenAI合作，开发新的AI助手",
          "谷歌在欧洲展开AI研究项目"
      ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

#2. 初始化模型并绑定工具
tool_list = [get_stock_price, search_news]
model_with_tools = model_openai.bind_tools(tool_list)

message_list = []
human_msg = HumanMessage(content="苹果公司今天的股价是多少？最近有什么样的新闻？")
message_list.append(human_msg)

#3. 工具调用
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)

    # 获取工具方法的列表
    tool_calls = response.tool_calls

    #如果列表为空，即为不存在的化，也就死模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    #如果有调用工具，处理工具调用响应
    # 4. 开发者根据模型的响应，调用工具并获取结果
    for call in tool_calls:
        if call["name"] == "get_stock_price_tool":
            tool_msg_stock = get_stock_price.invoke(call)
            message_list.append(tool_msg_stock)
        if call["name"] == "search_news_tool":
            tool_msg_news = search_news.invoke(call)
            message_list.append(tool_msg_news)

# 5. 调用大模型
final_response = model_openai.invoke(message_list)
message_list.append(final_response)

for msg in message_list:
    msg.pretty_print()


没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么样的新闻？
================================== Ai Message ==================================
Tool Calls:
  get_stock_price_tool (call_a20dccddbb60439492e2ed1c)
 Call ID: call_a20dccddbb60439492e2ed1c
  Args:
    company: 苹果公司
    timeframe: today
  search_news_tool (call_80825ffbcf8d4b7d9c5b5018)
 Call ID: call_80825ffbcf8d4b7d9c5b5018
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price_tool

苹果公司 today价格: 185.2美元
================================= Tool Message =================================
Name: search_news_tool

苹果发布新款iPhone，股价上涨3%
苹果与欧盟达成反垄断和解协议
苹果将在印度扩大生产规模
================================== Ai Message ==================================

苹果公司今天的股价是 **185.2美元**。

关于苹果公司的近期新闻包括：
*   苹果发布了新款iPhone，推动股价上涨了3%。
*   苹果与欧盟达成了反垄断和解协议。
*   苹果计划将在印度的生产规模进一步扩大。
================================== Ai Message ==